# Обучение модели для сервиса фрод-скоринга

Ноутбук обучает CatBoost на данных соревнования [teta-ml-1-2025](https://www.kaggle.com/competitions/teta-ml-1-2025)
и сохраняет модель в `fraud_detector/models/`, откуда её загружает сервис.

Предобработка **не написана здесь заново**, а импортируется из `fraud_detector/src/preprocessing.py`.
Поэтому признаки при обучении и в сервисе гарантированно одинаковые.

Перед запуском положите `train.csv` в папку `fraud_detector/train_data/`.

In [1]:
import sys
import json

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_recall_curve, f1_score

# Подключаем предобработку из сервиса
sys.path.append('../fraud_detector/src')
from preprocessing import run_preproc, CAT_FEATURES, FEATURES

RANDOM_STATE = 42

## 1. Загрузка данных

In [2]:
train = pd.read_csv('../fraud_detector/train_data/train.csv')
print('Размер:', train.shape)
print('Доля фрода:', round(train['target'].mean(), 4))
train.head()

Размер: (786431, 18)
Доля фрода: 0.0057


,transaction_time,merch,cat_id,amount,name_1,name_2,gender,street,one_city,us_state,post_code,lat,lon,population_city,jobs,merchant_lat,merchant_lon,target
0,2019-12-27 15:21,fraud_Cormier LLC,health_fitness,148.04,Daniel,Martinez,M,8510 Acevedo Burgs,Kent,OR,97033,45.0838,-120.6649,60,Museum education officer,45.042827,-120.709327,0
1,2019-04-17 23:09,"fraud_Brown, Homenick and Lesch",health_fitness,39.40,Grace,Williams,F,28812 Charles Mill Apt. 628,Plantersville,AL,36758,32.6176,-86.9475,1412,Drilling engineer,31.872266,-87.828247,0
2,2019-09-23 15:02,fraud_Ruecker-Mayert,kids_pets,52.96,Kyle,Park,M,7507 Larry Passage Suite 859,Mount Perry,OH,43760,39.8788,-82.1880,1831,Barrister's clerk,40.010874,-81.841249,0
3,2019-05-13 16:00,"fraud_Mante, Luettgen and Hackett",health_fitness,7.66,Monique,Martin,F,68276 Matthew Springs,Ratcliff,TX,75858,31.3833,-95.0619,43,"Engineer, production",30.888406,-95.141609,0
4,2019-08-18 07:27,fraud_Luettgen PLC,gas_transport,51.59,Christine,Johnson,F,8011 Chapman Tunnel Apt. 568,Blairsden-Graeagle,CA,96103,39.8127,-120.6405,1725,Chartered legal executive (England and Wales),39.376017,-121.311691,0


## 2. Предобработка

In [3]:
X = run_preproc(train)
y = train['target']

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print('Train:', X_train.shape, ' Valid:', X_valid.shape)
X.head()

Train: (629144, 15)  Valid: (157287, 15)


,merch,cat_id,gender,one_city,us_state,jobs,amount_log,population_city_log,lat,lon,merchant_lat,merchant_lon,distance_km,hour,day_of_week
0,fraud_Cormier LLC,health_fitness,M,Kent,OR,Museum education officer,5.004215,4.110874,45.0838,-120.6649,45.042827,-120.709327,5.738654,15,4
1,"fraud_Brown, Homenick and Lesch",health_fitness,F,Plantersville,AL,Drilling engineer,3.698830,7.253470,32.6176,-86.9475,31.872266,-87.828247,117.172182,23,2
2,fraud_Ruecker-Mayert,kids_pets,M,Mount Perry,OH,Barrister's clerk,3.988243,7.513164,39.8788,-82.1880,40.010874,-81.841249,33.007268,15,0
3,"fraud_Mante, Luettgen and Hackett",health_fitness,F,Ratcliff,TX,"Engineer, production",2.158715,3.784190,31.3833,-95.0619,30.888406,-95.141609,55.550169,16,0
4,fraud_Luettgen PLC,gas_transport,F,Blairsden-Graeagle,CA,Chartered legal executive (England and Wales),3.962526,7.453562,39.8127,-120.6405,39.376017,-121.311691,75.267261,7,6


## 3. Обучение CatBoost

Категориальные признаки CatBoost кодирует сам, поэтому передаём их списком в `cat_features`.
Обучение занимает несколько минут.

In [4]:
model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    eval_metric='AUC',
    random_seed=RANDOM_STATE,
    verbose=50,
)
model.fit(
    X_train, y_train,
    cat_features=CAT_FEATURES,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=50,
)

0:	test: 0.5741948	best: 0.5741948 (0)	total: 277ms	remaining: 2m 18s
50:	test: 0.9812632	best: 0.9812632 (50)	total: 8.76s	remaining: 1m 17s
100:	test: 0.9903561	best: 0.9903561 (100)	total: 17.4s	remaining: 1m 8s
150:	test: 0.9926881	best: 0.9926881 (150)	total: 26.1s	remaining: 1m
200:	test: 0.9939499	best: 0.9939499 (200)	total: 34.7s	remaining: 51.6s
250:	test: 0.9945425	best: 0.9945453 (247)	total: 43.1s	remaining: 42.7s
300:	test: 0.9949224	best: 0.9949224 (300)	total: 51.4s	remaining: 34s
350:	test: 0.9952124	best: 0.9952124 (350)	total: 59.9s	remaining: 25.4s
400:	test: 0.9952702	best: 0.9952983 (393)	total: 1m 8s	remaining: 16.8s
450:	test: 0.9955071	best: 0.9955138 (448)	total: 1m 16s	remaining: 8.31s
499:	test: 0.9958209	best: 0.9958234 (495)	total: 1m 24s	remaining: 0us

bestTest = 0.9958234124
bestIteration = 495

Shrink model to first 496 iterations.


## 4. Качество и подбор порога

Порог для `fraud_flag` выбираем так, чтобы F1 на валидации был максимальным.

In [5]:
valid_scores = model.predict_proba(X_valid)[:, 1]
print('ROC-AUC на валидации:', round(roc_auc_score(y_valid, valid_scores), 4))

precision, recall, thresholds = precision_recall_curve(y_valid, valid_scores)
f1 = 2 * precision * recall / (precision + recall + 1e-9)
best = np.argmax(f1[:-1])
threshold = float(thresholds[best])

print('Лучший порог:', round(threshold, 4))
print('F1 при этом пороге:', round(f1_score(y_valid, valid_scores > threshold), 4))

ROC-AUC на валидации: 0.9958
Лучший порог: 0.3266
F1 при этом пороге: 0.8541


## 5. Важность признаков

In [6]:
pd.Series(model.get_feature_importance(), index=FEATURES).sort_values(ascending=False)

amount_log             36.931359
hour                   21.750066
cat_id                 19.914003
merch                   6.888009
population_city_log     2.544373
one_city                2.492460
gender                  2.350908
jobs                    1.329501
distance_km             1.103869
day_of_week             1.059251
lat                     0.998201
us_state                0.765583
merchant_lat            0.681340
merchant_lon            0.664195
lon                     0.526880
dtype: float64

## 6. Сохранение модели для сервиса

In [7]:
model.save_model('../fraud_detector/models/model.cbm')

with open('../fraud_detector/models/model_config.json', 'w') as f:
    json.dump({'threshold': threshold, 'features': FEATURES}, f, indent=2)

print('Модель и порог сохранены в fraud_detector/models/')

Модель и порог сохранены в fraud_detector/models/


## 7. Проверка: скоринг одной транзакции

Так же, как это делает сервис: одна строка → предобработка → модель.

In [8]:
loaded = CatBoostClassifier()
loaded.load_model('../fraud_detector/models/model.cbm')

one_transaction = train.drop(columns='target').iloc[[0]]
score = loaded.predict_proba(run_preproc(one_transaction))[:, 1][0]
print('score =', round(score, 4), ' fraud_flag =', int(score > threshold))

score = 0.0002  fraud_flag = 0
